# PIX Participants Data Cleaning & Segmentation

## Project Overview

This project cleans, standardizes, and segments the official list of PIX participants published by the Central Bank of Brazil.

The objective is to prepare a high-quality dataset for business analysis, institution classification, and prospect segmentation.

## Load the Dataset

The official list of PIX participants is loaded into a pandas DataFrame. The dataset is then inspected to verify that it has been imported correctly before the data cleaning process begins.

In [45]:
import pandas as pd

# Load the dataset
df = pd.read_csv(
    "../data/Lista_Pix.csv",
    encoding="latin1",
    sep=";",
    header=1
)

# Display the first rows of the dataset
df.head()

,,Nome Reduzido,ISPB,CNPJ,Tipo de Instituição,Autorizada pelo BCB,Tipo de Participação no SPI,Tipo de Participação no Pix,Modalidade de Participação no Pix,Iniciação de Transação de Pagamento,Facilitador de serviço de Saque e Troco (FSS)
0,1,99PAY IP S.A.,24313102,24.313.102/0001-25,Instituição de Pagamento,Sim,Direta,Facultativa,Provedor de Conta Transacional,Sim,Não
1,2,A27 IP S/A,35534511,35.534.511/0001-78,Instituição de Pagamento,Sim,Indireta,Facultativa,Provedor de Conta Transacional,Não,Não
2,3,A55 SCD S.A.,48756121,48.756.121/0001-94,Sociedade de Crédito Direto,Sim,Direta,Facultativa,Provedor de Conta Transacional,Não,Não
3,4,ACCESSTAGE IP LTDA.,NaN,46.410.407/0001-98,Instituição de Pagamento,Sim,NaN,Facultativa,Iniciador,Sim,NaN
4,5,ACCREDITO SCD S.A.,37715993,37.715.993/0001-98,Sociedade de Crédito Direto,Sim,Direta,Facultativa,Provedor de Conta Transacional,Não,Não


## Initial Data Exploration

Before cleaning the dataset, an initial exploration is performed to understand its structure, dimensions, data types, and available variables.

This step provides an overview of the dataset and helps identify potential data quality issues.

In [46]:
# Display the dataset dimensions
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# Display general information about the dataset
df.info()

# Display the column names
print("\nColumn names:")
print(df.columns.tolist())

Rows: 919
Columns: 11
<class 'pandas.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 11 columns):
 #   Column                                         Non-Null Count  Dtype
---  ------                                         --------------  -----
 0                                                  919 non-null    str  
 1   Nome Reduzido                                  918 non-null    str  
 2   ISPB                                           899 non-null    str  
 3   CNPJ                                           918 non-null    str  
 4   Tipo de Instituição                            918 non-null    str  
 5   Autorizada pelo BCB                            918 non-null    str  
 6   Tipo de Participação no SPI                    899 non-null    str  
 7   Tipo de Participação no Pix                    918 non-null    str  
 8   Modalidade de Participação no Pix              918 non-null    str  
 9   Iniciação de Transação de Pagamento            918 non-null    st

## Data Quality Assessment

The dataset is assessed for potential data quality issues before any cleaning operations are performed.

Since this project focuses on data cleaning, segmentation, and business analysis rather than numerical calculations, the original data types are preserved.

The quality assessment includes:
- identifying duplicate records;
- inspecting duplicate observations before deciding whether they should be removed;
- checking for missing values.

In [47]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


No duplicate records were identified in the dataset.

### Missing Values

Missing values can affect data quality and lead to inaccurate analyses.

The dataset is inspected to identify whether any columns contain missing values before proceeding with the cleaning process.

In [48]:
# Count missing values in each column
missing_values = df.isna().sum()

print("Missing values by column:")
print(missing_values)

# Display the total number of missing values
print(f"\nTotal missing values: {missing_values.sum()}")

Missing values by column:
                                                   0
Nome Reduzido                                      1
ISPB                                              20
CNPJ                                               1
Tipo de Instituição                                1
Autorizada pelo BCB                                1
Tipo de Participação no SPI                       20
Tipo de Participação no Pix                        1
Modalidade de Participação no Pix                  1
Iniciação de Transação de Pagamento                1
Facilitador de serviço de Saque e Troco (FSS)    627
dtype: int64

Total missing values: 674


Missing values were identified primarily in SPI participation fields. These observations are intentionally preserved because they represent unknown information rather than data errors.

## Data Preparation

The original dataset is preserved to maintain the integrity of the official data published by the Central Bank of Brazil.

Missing values are intentionally left unchanged because they represent unknown information rather than incorrect data. In particular, institutions with missing values in the direct participation field will be treated as a separate business segment in a later stage of the analysis.

A copy of the original dataset is created for the remaining data preparation steps.

In [49]:
# Create a working copy of the original dataset
df_clean = df.copy()

# Display the first rows of the new DataFrame
df_clean.head()

,,Nome Reduzido,ISPB,CNPJ,Tipo de Instituição,Autorizada pelo BCB,Tipo de Participação no SPI,Tipo de Participação no Pix,Modalidade de Participação no Pix,Iniciação de Transação de Pagamento,Facilitador de serviço de Saque e Troco (FSS)
0,1,99PAY IP S.A.,24313102,24.313.102/0001-25,Instituição de Pagamento,Sim,Direta,Facultativa,Provedor de Conta Transacional,Sim,Não
1,2,A27 IP S/A,35534511,35.534.511/0001-78,Instituição de Pagamento,Sim,Indireta,Facultativa,Provedor de Conta Transacional,Não,Não
2,3,A55 SCD S.A.,48756121,48.756.121/0001-94,Sociedade de Crédito Direto,Sim,Direta,Facultativa,Provedor de Conta Transacional,Não,Não
3,4,ACCESSTAGE IP LTDA.,NaN,46.410.407/0001-98,Instituição de Pagamento,Sim,NaN,Facultativa,Iniciador,Sim,NaN
4,5,ACCREDITO SCD S.A.,37715993,37.715.993/0001-98,Sociedade de Crédito Direto,Sim,Direta,Facultativa,Provedor de Conta Transacional,Não,Não


## Feature Selection

The original dataset contains additional variables that are not required for the scope of this project.

To simplify the analysis, a new DataFrame is created containing only the variables that are relevant for institution profiling and business segmentation:

- Institution Name
- Institution Type
- Authorized by the Central Bank of Brazil
- SPI Participation Type
- Payment Initiation Service

These variables provide the information needed to support the subsequent business analysis.

In [50]:
# Select only the columns relevant for the analysis
df_clean = df[
    [
        "Nome Reduzido",
        "Tipo de Instituição",
        "Autorizada pelo BCB",
        "Tipo de Participação no SPI",
        "Iniciação de Transação de Pagamento"
    ]
].copy()

# Rename columns to English
df_clean.rename(
    columns={
        "Nome Reduzido": "Institution Name",
        "Tipo de Instituição": "Institution Type",
        "Autorizada pelo BCB": "Authorized by BCB",
        "Tipo de Participação no SPI": "SPI Participation Type",
        "Iniciação de Transação de Pagamento": "Payment Initiation Service"
    },
    inplace=True
)

# Display the new DataFrame
df_clean.head()

,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
0,99PAY IP S.A.,Instituição de Pagamento,Sim,Direta,Sim
1,A27 IP S/A,Instituição de Pagamento,Sim,Indireta,Não
2,A55 SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
3,ACCESSTAGE IP LTDA.,Instituição de Pagamento,Sim,NaN,Sim
4,ACCREDITO SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não


## Row Filtering

The dataset is filtered to retain only financial institutions authorized by the Central Bank of Brazil (BCB).

Since the focus of this project is on regulated financial institutions, records that are not authorized by the BCB are excluded from the analysis.

In [51]:
# Keep only institutions authorized by the Central Bank of Brazil
df_clean = df_clean[df_clean["Authorized by BCB"] == "Sim"].copy()

# Display the filtered DataFrame
df_clean.head()

,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
0,99PAY IP S.A.,Instituição de Pagamento,Sim,Direta,Sim
1,A27 IP S/A,Instituição de Pagamento,Sim,Indireta,Não
2,A55 SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
3,ACCESSTAGE IP LTDA.,Instituição de Pagamento,Sim,NaN,Sim
4,ACCREDITO SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não


## SPI Participation Filtering

After retaining only institutions authorized by the Central Bank of Brazil, the dataset is further refined based on each institution's SPI participation type.

Institutions with indirect SPI participation are excluded because they do not meet the project's target criteria. Institutions with direct SPI participation are retained, while records with missing SPI participation information are preserved for further investigation, as their participation status cannot be confirmed from the available data.

This filtering ensures that the final dataset focuses on confirmed direct SPI participants while retaining institutions that may represent potential business opportunities after additional validation.

In [52]:
# Keep direct SPI participants and institutions with missing SPI participation information
df_clean = df_clean[
    (df_clean["SPI Participation Type"] == "Direta") |
    (df_clean["SPI Participation Type"].isna())
].copy()

print(f"Remaining institutions: {len(df_clean)}")

df_clean.head()

Remaining institutions: 241


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
0,99PAY IP S.A.,Instituição de Pagamento,Sim,Direta,Sim
2,A55 SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
3,ACCESSTAGE IP LTDA.,Instituição de Pagamento,Sim,NaN,Sim
4,ACCREDITO SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
5,ACESSO SOLUÇÕES DE PAGAMENTO S.A. - INSTITUIÇÃ...,Instituição de Pagamento,Sim,Direta,Não


## SPI Participant Segmentation

The filtered dataset is divided into two groups based on SPI participation status.

Two separate DataFrames are created:

- **Direct SPI Participants**, containing institutions confirmed as direct participants in the Brazilian Instant Payment System (SPI);
- **Unknown SPI Participation**, containing institutions with missing SPI participation information.

The second group is intentionally preserved because the available data does not confirm whether these institutions participate directly in the SPI. These records may require additional validation or support a separate business outreach strategy.

In [53]:
# Create separate DataFrames based on SPI participation status

df_direct_spi = df_clean[
    df_clean["SPI Participation Type"] == "Direta"
].copy()

df_unknown_spi = df_clean[
    df_clean["SPI Participation Type"].isna()
].copy()

print(f"Direct SPI participants: {len(df_direct_spi)}")
print(f"Unknown SPI participation: {len(df_unknown_spi)}")

display(df_direct_spi.head())
display(df_unknown_spi.head())

Direct SPI participants: 222
Unknown SPI participation: 19


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
0,99PAY IP S.A.,Instituição de Pagamento,Sim,Direta,Sim
2,A55 SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
4,ACCREDITO SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
5,ACESSO SOLUÇÕES DE PAGAMENTO S.A. - INSTITUIÇÃ...,Instituição de Pagamento,Sim,Direta,Não
8,ADYEN DO BRASIL IP LTDA.,Instituição de Pagamento,Sim,Direta,Sim


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
3,ACCESSTAGE IP LTDA.,Instituição de Pagamento,Sim,NaN,Sim
27,B3 IP LTDA,Instituição de Pagamento,Sim,NaN,Sim
90,BELVO IP,Instituição de Pagamento,Sim,NaN,Sim
653,CREDSYSTEM IP,Instituição de Pagamento,Sim,NaN,Sim
658,CRYSTAL IP,Instituição de Pagamento,Sim,NaN,Sim


## Payment Initiation Service Segmentation

Within Payall's commercial strategy, financial institutions that operate as **Originating Institutions (OIs)** are classified as **Tier 1** prospects.

To support targeted marketing and sales initiatives, both SPI participant groups are further segmented according to their Payment Initiation Service (PIS) status.

Four DataFrames are created:

- Direct SPI participants with PIS enabled;
- Direct SPI participants without PIS;
- Institutions with unknown SPI participation and PIS enabled;
- Institutions with unknown SPI participation and no PIS.

This segmentation allows marketing campaigns to be tailored according to both SPI participation status and payment initiation capabilities, while preserving institutions whose SPI participation still requires further validation.

In [54]:
# Segment institutions by Payment Initiation Service (PIS) status

# Confirmed direct SPI participants
df_direct_spi_pis_yes = df_direct_spi[
    df_direct_spi["Payment Initiation Service"] == "Sim"
].copy()

df_direct_spi_pis_no = df_direct_spi[
    df_direct_spi["Payment Initiation Service"] == "Não"
].copy()

# Institutions with unknown SPI participation
df_unknown_spi_pis_yes = df_unknown_spi[
    df_unknown_spi["Payment Initiation Service"] == "Sim"
].copy()

df_unknown_spi_pis_no = df_unknown_spi[
    df_unknown_spi["Payment Initiation Service"] == "Não"
].copy()

print(f"Direct SPI + PIS: {len(df_direct_spi_pis_yes)}")
print(f"Direct SPI + No PIS: {len(df_direct_spi_pis_no)}")
print(f"Unknown SPI + PIS: {len(df_unknown_spi_pis_yes)}")
print(f"Unknown SPI + No PIS: {len(df_unknown_spi_pis_no)}")

display(df_direct_spi_pis_yes.head())
display(df_direct_spi_pis_no.head())
display(df_unknown_spi_pis_yes.head())
display(df_unknown_spi_pis_no.head())

Direct SPI + PIS: 57
Direct SPI + No PIS: 155
Unknown SPI + PIS: 17
Unknown SPI + No PIS: 0


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
0,99PAY IP S.A.,Instituição de Pagamento,Sim,Direta,Sim
8,ADYEN DO BRASIL IP LTDA.,Instituição de Pagamento,Sim,Direta,Sim
16,ASA SCFI S.A.,"Sociedade de Crédito, Financiamento e Investim...",Sim,Direta,Sim
24,AVANCARD PROVER IP LTDA,Instituição de Pagamento,Sim,Direta,Sim
30,BANCO BTG PACTUAL S.A.,Banco Múltiplo,Sim,Direta,Sim


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
2,A55 SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
4,ACCREDITO SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
5,ACESSO SOLUÇÕES DE PAGAMENTO S.A. - INSTITUIÇÃ...,Instituição de Pagamento,Sim,Direta,Não
11,ALL IN CRED SCD S.A.,Sociedade de Crédito Direto,Sim,Direta,Não
15,ARTTA SCD,Sociedade de Crédito Direto,Sim,Direta,Não


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service
3,ACCESSTAGE IP LTDA.,Instituição de Pagamento,Sim,NaN,Sim
27,B3 IP LTDA,Instituição de Pagamento,Sim,NaN,Sim
90,BELVO IP,Instituição de Pagamento,Sim,NaN,Sim
653,CREDSYSTEM IP,Instituição de Pagamento,Sim,NaN,Sim
658,CRYSTAL IP,Instituição de Pagamento,Sim,NaN,Sim


,Institution Name,Institution Type,Authorized by BCB,SPI Participation Type,Payment Initiation Service


## Segment Review

The segmentation results identified four potential business groups.

However, no institutions were found in the **Unknown SPI + No PIS** segment. Since this segment contains no observations, it is excluded from the remaining analysis and visualizations.

The final business segmentation therefore consists of three actionable groups.

In [56]:
# Exclude the empty business segment from the final analysis

segment_summary = pd.DataFrame({
    "Segment": [
        "Direct SPI + PIS",
        "Direct SPI + No PIS",
        "Unknown SPI + PIS"
    ],
    "Institutions": [
        len(df_direct_spi_pis_yes),
        len(df_direct_spi_pis_no),
        len(df_unknown_spi_pis_yes)
    ]
})

display(segment_summary)

,Segment,Institutions
0,Direct SPI + PIS,57
1,Direct SPI + No PIS,155
2,Unknown SPI + PIS,17


## Executive Summary

The data cleaning and segmentation process resulted in three actionable business segments.

The largest group consists of confirmed Direct SPI participants without Payment Initiation Service (PIS), followed by Direct SPI participants that operate as Payment Initiation Service Providers. A smaller group includes institutions with unknown SPI participation status but confirmed Payment Initiation Service.

These segments provide a structured view of the market and can support commercial prioritization and targeted outreach strategies.

In [59]:
import plotly.express as px

fig = px.bar(
    segment_summary,
    x="Segment",
    y="Institutions",
    text="Institutions",
    title="Distribution of Institutions by Business Segment"
)

fig.update_traces(textposition="outside")

fig.update_layout(
    title_x=0.5,
    xaxis_title="Business Segment",
    yaxis_title="Number of Institutions",
    template="plotly_white"
)

fig.show()

## Conclusion

This project transformed the official Central Bank of Brazil PIX participants dataset into a structured, business-ready dataset through data cleaning, feature selection, and rule-based segmentation.

The final segmentation identified three actionable groups of financial institutions that can support institution profiling, commercial prioritization, and targeted outreach strategies.

Overall, this project demonstrates how a systematic data preparation process can convert raw regulatory data into meaningful business insights.